# Lock-in Intervention LLM — QLoRA Fine-Tune
**Model:** Qwen 2.5 7B Instruct  
**Method:** QLoRA (4-bit) via Unsloth  
**Hardware:** A100 (40 GB)  
**Dataset:** `intervention_train.jsonl` / `intervention_eval.jsonl`

Upload both JSONL files to your Google Drive before running.

In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────────
# Unsloth gives ~2x faster training and ~50% less VRAM vs plain HF on A100.
# Run once; runtime will restart automatically after install.

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install -q datasets

In [ ]:
# ── CELL 2: Mount Drive and set paths ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ▼ Change this to where you uploaded the two JSONL files in your Drive
DATA_DIR = "/content/drive/MyDrive/lockin_training"

TRAIN_FILE = os.path.join(DATA_DIR, "intervention_train.jsonl")
EVAL_FILE  = os.path.join(DATA_DIR, "intervention_eval.jsonl")

assert os.path.exists(TRAIN_FILE), f"Train file not found: {TRAIN_FILE}"
assert os.path.exists(EVAL_FILE),  f"Eval file not found:  {EVAL_FILE}"
print(f"Train: {TRAIN_FILE}")
print(f"Eval:  {EVAL_FILE}")

In [ ]:
# ── CELL 3: Load model + configure QLoRA ────────────────────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048   # All examples fit within this; change to 3072 if needed
DTYPE          = None   # Auto-detect; A100 will use bfloat16
LOAD_IN_4BIT   = True   # QLoRA: 4-bit base weights

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

# Apply QLoRA adapter
# rank=32 is a good balance for A100; use rank=64 for maximum quality.
model = FastLanguageModel.get_peft_model(
    model,
    r                    = 32,
    target_modules       = ["q_proj", "k_proj", "v_proj", "o_proj",
                             "gate_proj", "up_proj", "down_proj"],
    lora_alpha           = 64,      # 2× rank is the standard starting point
    lora_dropout         = 0.05,
    bias                 = "none",
    use_gradient_checkpointing = "unsloth",  # reduces VRAM further
    random_state         = 42,
    use_rslora           = True,    # rank-stabilised LoRA — more stable training
)

model.print_trainable_parameters()

In [ ]:
# ── CELL 4: Load and inspect dataset ────────────────────────────────────────
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "eval": EVAL_FILE},
)

print(dataset)
print("\n── Sample training example (first 600 chars) ────────────────────")
print(dataset["train"][0]["text"][:600])
print("...[truncated]")

# Verify token lengths — all must be within MAX_SEQ_LENGTH
print("\n── Token length check ───────────────────────────────────────────")
sample_lengths = [
    len(tokenizer(ex["text"])["input_ids"])
    for ex in dataset["train"].select(range(min(100, len(dataset["train"]))))
]
print(f"  First 100 examples — avg={sum(sample_lengths)//len(sample_lengths)} "
      f"max={max(sample_lengths)} tokens")

In [ ]:
# ── CELL 5: Configure trainer ────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUTPUT_DIR = "/content/drive/MyDrive/lockin_training/checkpoints"

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = dataset["train"],
    eval_dataset    = dataset["eval"],
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing          = False,  # keep False — examples have varying lengths
    args = TrainingArguments(
        # ── Core hyperparameters ──────────────────────────────────────────
        num_train_epochs          = 3,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,   # effective batch = 16
        learning_rate             = 2e-4,
        lr_scheduler_type         = "cosine",
        warmup_ratio              = 0.05,
        weight_decay              = 0.01,
        optim                     = "adamw_8bit",
        # ── Precision ────────────────────────────────────────────────────
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),     # A100 supports bfloat16 — use it
        # ── Logging & saving ─────────────────────────────────────────────
        logging_steps             = 20,
        eval_strategy             = "steps",
        eval_steps                = 100,
        save_strategy             = "steps",
        save_steps                = 200,
        save_total_limit          = 3,
        load_best_model_at_end    = True,
        metric_for_best_model     = "eval_loss",
        output_dir                = OUTPUT_DIR,
        report_to                 = "none",   # change to "wandb" if you use W&B
        seed                      = 42,
        # ── A100 optimisations ───────────────────────────────────────────
        max_grad_norm             = 1.0,
        dataloader_num_workers    = 2,
    ),
)

print("Trainer configured.")
print(f"  Steps per epoch: {len(dataset['train']) // (4 * 4)}")
print(f"  Total steps:     {len(dataset['train']) // (4 * 4) * 3}")

In [ ]:
# ── CELL 6: Train ───────────────────────────────────────────────────────────
# On A100 40GB, expect ~25-35 minutes for 3 epochs over 1310 examples.

trainer_stats = trainer.train()

print("\n── Training complete ───────────────────────────────────────────")
print(f"  Runtime:     {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"  Train loss:  {trainer_stats.metrics['train_loss']:.4f}")
print(f"  Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.1f}")

In [ ]:
# ── CELL 7: Automatic evaluation ────────────────────────────────────────────
# Runs the fine-tuned model on the 145 eval examples and scores:
#   1. JSON validity rate      — does the model output parseable JSON?
#   2. Field completeness      — are all 4 required fields present?
#   3. Logic accuracy          — does intervene match cooldown_status?
#   4. Schema accuracy         — does content match the expected schema?
#   5. Content null rate       — content=null only when tier=none?

import json
from collections import Counter

FastLanguageModel.for_inference(model)  # enable fast inference mode

REQUIRED_FIELDS    = {"intervene", "tier", "type", "content"}
VALID_TIERS        = {"subtle", "moderate", "strong", "special", "none"}
VALID_TYPES        = {
    "focus_point", "ambient_sound", "chime", "re_engagement",
    "comprehension_check", "section_summary", "break_suggestion",
    "gamification", "text_reformat", "none"
}
TYPE_REQUIRED_KEYS = {
    "focus_point":         ["prompt"],
    "ambient_sound":       ["sound_type", "label"],
    "chime":               ["chime_type", "message"],
    "re_engagement":       ["headline", "body", "cta"],
    "comprehension_check": ["type", "question"],
    "section_summary":     ["title", "summary", "key_point"],
    "break_suggestion":    ["headline", "message", "duration_minutes"],
    "gamification":        ["event_type", "xp_awarded", "message"],
    "text_reformat":       ["mode"],
}

results = []

print("Running eval inference... (this will take a few minutes)")
for i, ex in enumerate(dataset["eval"]):
    full_text   = ex["text"]
    # Strip the assistant turn — feed only system + user as prompt
    prompt_end  = full_text.rfind("<|im_start|>assistant") + len("<|im_start|>assistant\n")
    prompt      = full_text[:prompt_end]

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens   = 400,
            temperature      = 0.1,   # low temp for structured JSON output
            do_sample        = True,
            pad_token_id     = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Parse the ground-truth output from the full example text
    asst_start = full_text.rfind("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
    ground_truth_str = full_text[asst_start:].replace("<|im_end|>", "").strip()
    gt = json.loads(ground_truth_str)

    # Parse generated
    pred = None
    try:
        pred = json.loads(generated)
        valid_json = True
    except Exception:
        valid_json = False

    fields_ok  = valid_json and REQUIRED_FIELDS.issubset(pred.keys())
    tier_ok    = fields_ok and pred["tier"] in VALID_TIERS
    type_ok    = fields_ok and pred["type"] in VALID_TYPES

    # Logic: intervene must be True iff cooldown=clear and tier!=none
    logic_ok = False
    if fields_ok:
        logic_ok = (pred["intervene"] == gt["intervene"])

    # Schema: content has required keys for its type
    schema_ok = False
    if fields_ok:
        ptype   = pred.get("type", "")
        content = pred.get("content")
        if ptype == "none":
            schema_ok = content is None
        elif ptype in TYPE_REQUIRED_KEYS and isinstance(content, dict):
            schema_ok = all(k in content for k in TYPE_REQUIRED_KEYS[ptype])

    results.append({
        "valid_json": valid_json,
        "fields_ok":  fields_ok,
        "tier_ok":    tier_ok,
        "type_ok":    type_ok,
        "logic_ok":   logic_ok,
        "schema_ok":  schema_ok,
        "pred":       pred,
        "gt":         gt,
    })

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(dataset['eval'])} examples evaluated...")

n = len(results)
print(f"\n══ EVALUATION RESULTS ({n} examples) ════════════════════════")
print(f"  JSON valid:       {sum(r['valid_json'] for r in results)}/{n}  ({sum(r['valid_json'] for r in results)/n*100:.1f}%)")
print(f"  Fields complete:  {sum(r['fields_ok']  for r in results)}/{n}  ({sum(r['fields_ok']  for r in results)/n*100:.1f}%)")
print(f"  Tier valid:       {sum(r['tier_ok']    for r in results)}/{n}  ({sum(r['tier_ok']    for r in results)/n*100:.1f}%)")
print(f"  Type valid:       {sum(r['type_ok']    for r in results)}/{n}  ({sum(r['type_ok']    for r in results)/n*100:.1f}%)")
print(f"  Logic correct:    {sum(r['logic_ok']   for r in results)}/{n}  ({sum(r['logic_ok']   for r in results)/n*100:.1f}%)  (intervene matches cooldown)")
print(f"  Schema correct:   {sum(r['schema_ok']  for r in results)}/{n}  ({sum(r['schema_ok']  for r in results)/n*100:.1f}%)  (content keys valid)")

# Show 3 failure cases
failures = [r for r in results if not r["valid_json"] or not r["logic_ok"]]
if failures:
    print(f"\n── Failure samples ({min(3,len(failures))} shown) ─────────────────")
    for r in failures[:3]:
        print(f"  GT:   {json.dumps(r['gt'])}")
        print(f"  Pred: {str(r['pred'])[:200]}")
        print()
else:
    print("\n  No failures — all predictions are valid and logically correct.")

In [ ]:
# ── CELL 8: Manual spot-check (qualitative review) ──────────────────────────
# Run the model on 3 hand-crafted inputs covering different scenarios.
# This is the most meaningful evaluation for a generative task.

SYSTEM_PROMPT = """You are an adaptive reading assistant engine embedded in a digital reading tool called Lock-in. Every 10 seconds you receive a 30-second window of signals about a student's attentional state, drift trajectory, and the text they are currently reading. Your task is to: (1) identify the most appropriate intervention type and tier (subtle | moderate | strong | special) based on the signals, and always generate its content; (2) set 'intervene' to true only when cooldown_status is 'clear' — if cooldown_status is 'cooling', set 'intervene' to false but still output the full content of what you would have fired, so the system can schedule it for the next clear window; (3) if no intervention is warranted at all (tier='none'), set 'intervene' to false and 'content' to null. Respond with a single valid JSON object only — no prose, no markdown fences."""

TEST_CASES = [
    {
        "label": "Heavy drift, strong section summary expected",
        "input": {
            "session_context": {"elapsed_minutes": 7.5, "session_stage": "mid",
                "last_intervention": {"type": "chime", "tier": "subtle", "seconds_ago": 180},
                "cooldown_status": "clear", "xp": 320, "badges_earned": ["first_focus_streak"]},
            "attentional_state_window": [
                {"primary_state": "cognitive_overload", "confidence": 0.72,
                 "distribution": {"focused": 0.1, "drifting": 0.18, "hyperfocused": 0.0, "cognitive_overload": 0.72}},
                {"primary_state": "cognitive_overload", "confidence": 0.68,
                 "distribution": {"focused": 0.12, "drifting": 0.2, "hyperfocused": 0.0, "cognitive_overload": 0.68}},
                {"primary_state": "cognitive_overload", "confidence": 0.81,
                 "distribution": {"focused": 0.05, "drifting": 0.14, "hyperfocused": 0.0, "cognitive_overload": 0.81}},
            ],
            "drift_progression": {"drift_level": [0.55, 0.68, 0.74], "engagement_score": [0.35, 0.31, 0.28], "drift_ema": 0.69},
            "user_baseline": {"wpm_effective": 280, "idle_ratio_mean": 0.35, "regress_rate_mean": 0.02, "para_dwell_median_s": 20},
            "reading_context": {
                "current_paragraph_index": 42,
                "text_window": [
                    "Working memory capacity is often cited as the primary bottleneck in complex cognitive tasks. Unlike long-term memory, which has virtually unlimited storage, working memory can hold only a small number of items at once, typically estimated at around four chunks of information.",
                    "The concept of cognitive load theory, developed by John Sweller in the late 1980s, proposes that learning is most effective when the total cognitive demand placed on the learner does not exceed the limits of working memory. Instructional designers must therefore minimise extraneous load while maximising germane load.",
                    "Split-attention effect occurs when learners must mentally integrate multiple sources of information that are physically or temporally separated. This imposes additional extraneous cognitive load and can be mitigated through techniques such as integrated formats or contiguity principles."
                ]
            }
        }
    },
    {
        "label": "Sustained focus, comprehension check expected (clear)",
        "input": {
            "session_context": {"elapsed_minutes": 6.0, "session_stage": "mid",
                "last_intervention": {"type": "gamification", "tier": "subtle", "seconds_ago": 210},
                "cooldown_status": "clear", "xp": 410, "badges_earned": ["first_focus_streak", "page_turner"]},
            "attentional_state_window": [
                {"primary_state": "focused", "confidence": 0.88, "distribution": {"focused": 0.88, "drifting": 0.08, "hyperfocused": 0.02, "cognitive_overload": 0.02}},
                {"primary_state": "focused", "confidence": 0.91, "distribution": {"focused": 0.91, "drifting": 0.06, "hyperfocused": 0.02, "cognitive_overload": 0.01}},
                {"primary_state": "focused", "confidence": 0.86, "distribution": {"focused": 0.86, "drifting": 0.10, "hyperfocused": 0.03, "cognitive_overload": 0.01}},
            ],
            "drift_progression": {"drift_level": [0.08, 0.05, 0.04], "engagement_score": [0.82, 0.85, 0.88], "drift_ema": 0.05},
            "user_baseline": {"wpm_effective": 310, "idle_ratio_mean": 0.28, "regress_rate_mean": 0.01, "para_dwell_median_s": 18},
            "reading_context": {
                "current_paragraph_index": 28,
                "text_window": [
                    "The spacing effect refers to the empirical finding that information is better remembered when learning is distributed across multiple sessions rather than concentrated in a single massed session. This effect has been observed consistently across decades of research.",
                    "Interleaved practice, in which different topics or problem types are mixed together during study, has been shown to produce superior long-term retention compared to blocked practice, despite feeling less productive to learners in the moment.",
                    "Retrieval practice, or the testing effect, demonstrates that actively recalling information from memory produces stronger memory traces than passively re-reading the same material. Even a single retrieval attempt can significantly improve later test performance."
                ]
            }
        }
    },
    {
        "label": "Cooling + single drift — content generated but suppressed",
        "input": {
            "session_context": {"elapsed_minutes": 4.0, "session_stage": "early",
                "last_intervention": {"type": "re_engagement", "tier": "moderate", "seconds_ago": 45},
                "cooldown_status": "cooling", "xp": 175, "badges_earned": ["first_focus_streak"]},
            "attentional_state_window": [
                {"primary_state": "focused", "confidence": 0.82, "distribution": {"focused": 0.82, "drifting": 0.14, "hyperfocused": 0.01, "cognitive_overload": 0.03}},
                {"primary_state": "focused", "confidence": 0.78, "distribution": {"focused": 0.78, "drifting": 0.18, "hyperfocused": 0.01, "cognitive_overload": 0.03}},
                {"primary_state": "drifting",  "confidence": 0.61, "distribution": {"focused": 0.31, "drifting": 0.61, "hyperfocused": 0.0, "cognitive_overload": 0.08}},
            ],
            "drift_progression": {"drift_level": [0.12, 0.22, 0.41], "engagement_score": [0.75, 0.68, 0.55], "drift_ema": 0.35},
            "user_baseline": {"wpm_effective": 250, "idle_ratio_mean": 0.40, "regress_rate_mean": 0.015, "para_dwell_median_s": 22},
            "reading_context": {
                "current_paragraph_index": 15,
                "text_window": [
                    "Selective attention allows humans to focus on a specific stimulus while filtering out competing distractions. This capacity is not unlimited, however, and extended periods of directed attention lead to a gradual depletion of attentional resources.",
                    "Attention restoration theory proposes that natural environments replenish directed attentional resources by engaging involuntary attention, which does not deplete in the same way. Exposure to natural settings has been associated with improved performance on tasks requiring sustained focus."
                ]
            }
        }
    },
]

for tc in TEST_CASES:
    print(f"\n{'═'*60}")
    print(f"SCENARIO: {tc['label']}")
    print('═'*60)

    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{json.dumps(tc['input'], separators=(',',':'))}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=400, temperature=0.1,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True).strip()
    print("MODEL OUTPUT:")
    try:
        parsed = json.loads(generated)
        print(json.dumps(parsed, indent=2))
    except Exception:
        print("[INVALID JSON]:", generated[:400])

In [ ]:
# ── CELL 9: Save LoRA adapter + export to GGUF for Ollama ───────────────────
# Saves in two forms:
#   (a) HuggingFace format (LoRA adapter only) — for future merging/fine-tuning
#   (b) GGUF Q4_K_M — the format Ollama loads directly

ADAPTER_SAVE_PATH = "/content/drive/MyDrive/lockin_training/lockin_intervention_adapter"
GGUF_SAVE_PATH    = "/content/drive/MyDrive/lockin_training/lockin_intervention_gguf"

# Save the LoRA adapter weights (fast, small)
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"LoRA adapter saved to: {ADAPTER_SAVE_PATH}")

# Merge adapter into base weights and export as GGUF Q4_K_M
# Q4_K_M is the recommended quantisation for Ollama — good quality/speed balance
model.save_pretrained_gguf(
    GGUF_SAVE_PATH,
    tokenizer,
    quantization_method = "q4_k_m",
)
print(f"GGUF model saved to:   {GGUF_SAVE_PATH}")
print("\nTo use with Ollama, create a Modelfile:")
print("  FROM ./lockin_intervention_gguf/unsloth.Q4_K_M.gguf")
print("  PARAMETER temperature 0.1")
print("  PARAMETER stop '<|im_end|>'")
print("Then: ollama create lockin-intervention -f Modelfile")

In [ ]:
# ── CELL 10: Create Ollama Modelfile ────────────────────────────────────────
# After downloading the GGUF from Drive, use this Modelfile locally.

SYSTEM_PROMPT_FOR_OLLAMA = """You are an adaptive reading assistant engine embedded in a digital reading tool called Lock-in. Every 10 seconds you receive a 30-second window of signals about a student's attentional state, drift trajectory, and the text they are currently reading. Your task is to: (1) identify the most appropriate intervention type and tier (subtle | moderate | strong | special) based on the signals, and always generate its content; (2) set 'intervene' to true only when cooldown_status is 'clear' — if cooldown_status is 'cooling', set 'intervene' to false but still output the full content of what you would have fired, so the system can schedule it for the next clear window; (3) if no intervention is warranted at all (tier='none'), set 'intervene' to false and 'content' to null. Respond with a single valid JSON object only — no prose, no markdown fences."""

modelfile = f"""FROM ./unsloth.Q4_K_M.gguf

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
PARAMETER num_ctx 2048

SYSTEM \"\"\"
{SYSTEM_PROMPT_FOR_OLLAMA}
\"\"\"
"""

modelfile_path = "/content/drive/MyDrive/lockin_training/Modelfile"
with open(modelfile_path, "w") as f:
    f.write(modelfile)

print(f"Modelfile written to: {modelfile_path}")
print("\nOnce downloaded locally, run:")
print("  ollama create lockin-intervention -f Modelfile")
print("  ollama run lockin-intervention")